# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

# 0. Document Instructions

## 0.1. Git Version Control

After making modifications, please use Colab's function of "Save in Github to keep changes" (Github authorization required) to commit the changes to the repository (Your own branch if needed).

**NOTE**: When A modifies this notebook, but have not commited to Github, if B opens this notebook, they will see the Github old version.

## 0.2. Handle persistent data storage

Mount Google Drive instead of relying on Colab's temporary run-time storage.

**IMPORTANT:** Although Google Colab allows collaboration of the notebook, data and folders are originally in Junxuan's drive, this could cause issues when refering to the mounted directory path. To allow the code to work for all of us, (who's not junxuan) please create a **short-cut** of Junxuan's shared folder in our own Google Drive, path `MyDrive/`.

In [ ]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

## 0.3 What this notebook does

Runs the whole pipeline for every model in one go, then the model comparison:

| Order | Notebook | Saves to |
|---|---|---|
| 1 | Step 1 `data_exploration` | `evaluation_runs/<model>/processed/` |
| 2 | Step 2 `extraction_verification`; for the new models up to 2.4, because 2.5 needs the manual labels of gpt-4o-mini | `evaluation_runs/<model>/processed/` |
| 3 | Task 3 `transparency_reasoning` | `evaluation_runs/<model>/processed/` |
| 4 | Task 4 `bias_n_fairness` | `evaluation_runs/<model>/results/` |
| 5 | Task 6 `evaluation_comparison`, after all models | `evaluation_runs/comparison/` |

Each notebook runs as it is in the repository. Only two things change in the copy that runs: `MODEL` is set to the model of the run, and the Drive mount is skipped because this notebook has mounted the drive already. Every executed notebook, with its outputs and plots, is saved in `evaluation_runs/<model>/results/` (Task 6 in `evaluation_runs/comparison/`).

**Before running:** the manual labels `manual_verifier_sample_30_annotated.xlsx` must be in `evaluation_runs/gpt-4o-mini/processed/`. If a notebook fails, the run stops and the failed notebook is saved with its error in the same place.

In [ ]:
%pip install -q nbclient nbformat

In [ ]:
import subprocess
from pathlib import Path

import nbformat
import pandas as pd
from nbclient import NotebookClient

project_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data")
runs_dir = project_dir / "evaluation_runs"

# the pipeline notebooks clone this branch for their helpers, so they are run from it too
repo_url = "https://github.com/lay-animals-projects/food-recommender-evaluation.git"
repo_branch = "anqizhou"
repo_dir = Path("/content/food-recommender-evaluation")

# models to run, in this order
MODELS = ["gpt-4o-mini", "gpt-oss-120b", "qwen3.8-27b"]

# the pipeline of one model: name of the saved copy -> notebook in the repository
PIPELINE = {
    "1_data_exploration": "notebooks/data_exploration/data_exploration.ipynb",
    "2_extraction_verification": "notebooks/extraction_verification/extraction_verification.ipynb",
    "3_transparency_reasoning": "notebooks/transparency_reasoning.ipynb",
    "4_bias_n_fairness": "notebooks/bias_n_fairness.ipynb",
}
COMPARISON = "notebooks/evaluation_comparison.ipynb"

In [ ]:
# get the notebooks: clone the repository, or update the copy from an earlier run
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", repo_branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", repo_branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", repo_branch], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(repo_dir)], check=True)

print(subprocess.run(["git", "-C", str(repo_dir), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# the inputs every run needs
inputs = {
    "personas": project_dir / "raw" / "personas.json",
    "recipes": project_dir / "raw" / "recipes.parquet",
    "gpt-4o-mini explanations": project_dir / "raw" / "recommendations.json",
    "gpt-oss-120b explanations": project_dir / "llm_results" / "GPT-OSS-120B-output" / "recommendations.json",
    "qwen3.8-27b explanations": project_dir / "llm_results" / "Qwen3.8-27B-output" / "recommendations.json",
    "manual labels (2.5)": runs_dir / "gpt-4o-mini" / "processed" / "manual_verifier_sample_30_annotated.xlsx",
}
missing = {name: path for name, path in inputs.items() if not path.exists()}
for name, path in missing.items():
    print("missing:", name, "->", path)
assert not missing, "put the missing files in place first"
print("all inputs found")

## 1. Run the notebooks

In [ ]:
MODEL_LINE = 'MODEL = "gpt-4o-mini"'


def prepare(notebook, model=None):
    """A copy of a repository notebook, set up for this run."""
    nb = nbformat.read(repo_dir / notebook, as_version=4)

    for cell in nb.cells:
        # the drive is already mounted by this notebook
        if "drive.mount(" in cell.source:
            cell.source = "# drive already mounted by run_pipeline"

    if model is not None:
        # the model of this run
        switches = [cell for cell in nb.cells if MODEL_LINE in cell.source]
        assert len(switches) == 1, f"{notebook} has no MODEL switch"
        switches[0].source = switches[0].source.replace(MODEL_LINE, f'MODEL = "{model}"')

        # 2.5 validates the verifier with the manual labels, which exist only for gpt-4o-mini
        headings = [k for k, cell in enumerate(nb.cells) if cell.cell_type == "markdown" and cell.source.startswith("## 2.5")]
        if model != "gpt-4o-mini" and headings:
            nb.cells = nb.cells[:headings[0]]
    return nb


def run(nb, saved_as):
    """Execute a prepared notebook; save it with its outputs, also when a cell fails."""
    client = NotebookClient(nb, timeout=3600, kernel_name="python3", resources={"metadata": {"path": "/content"}})
    try:
        client.execute()
    finally:
        saved_as.parent.mkdir(parents=True, exist_ok=True)
        nbformat.write(nb, saved_as)

In [ ]:
# Step 1, Step 2, Task 3 and Task 4 for every model
for model in MODELS:
    for name, notebook in PIPELINE.items():
        saved_as = runs_dir / model / "results" / f"{name}.ipynb"
        print(f"{model}: {name} ...", flush=True)
        run(prepare(notebook, model), saved_as)
    print(f"{model}: done", flush=True)

In [ ]:
# Task 6 over all models
saved_as = runs_dir / "comparison" / "evaluation_comparison.ipynb"
run(prepare(COMPARISON), saved_as)
print("comparison saved in", saved_as.parent)

## 2. Check the results

In [ ]:
# a short check per model
rows = []
for model in MODELS:
    processed = runs_dir / model / "processed"
    claims = pd.read_parquet(processed / "verified_claims.parquet")
    levels = pd.read_parquet(processed / "transparency_levels.parquet")
    rows.append({
        "model": model,
        "explanations": len(levels),
        "claims": len(claims),
        "hallucinated explanations": claims.loc[claims["verification_status"].eq("contradicted"), "recommendation_id"].nunique(),
        "High / Medium / Low": " / ".join(str(int(levels["level"].eq(level).sum())) for level in ["High", "Medium", "Low"]),
    })
display(pd.DataFrame(rows))